# Camada Silver

A camada Silver recebe a tabela da camada Bronze (o `history.csv` original, sem alterações) e o diagnóstico de qualidade feito lá. Ela corrige os problemas encontrados e entrega uma base limpa, padronizada e com os tipos corretos. É dela que partem a estatística descritiva e as três Golds.

Cada decisão de tratamento vem acompanhada do motivo, para que outra pessoa consiga repetir o processo e chegar no mesmo resultado.

Do ponto de vista de segurança, a Silver é uma camada de acesso restrito: ela ainda tem o ID real do cliente e o texto do feedback. As Golds, usadas para análise, não carregam essas informações.

## 1. Leitura

In [1]:
import pandas as pd
import numpy as np

df_bronze = pd.read_csv('../Data/Bronze/tabela_bronze.csv')

# Trabalhamos sempre numa cópia para não alterar a base original
df = df_bronze.copy(deep=True)

print('Linhas e colunas:', df.shape)
df.head(3)

Linhas e colunas: (5634, 23)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,CustomerFeedback,MonthlyIncome
0,4578-PHJYZ,Male,0,Yes,Yes,52,Yes,No,DSL,No,Yes,Yes,Yes,Yes,No,One year,Yes,Electronic check,68.75,3482.85,No,"I have been a customer with this internet provider for over four years now and I have been satisfied with the service. The DSL internet is reliable and I have had no issues with connectivity. The monthly charges of $68.75 are reasonable for the service I receive. I appreciate the convenience of paying with electronic check. Overall, I am happy with my experience and have no plans to switch providers.",6532
1,6289-CPNLD,Male,0,Yes,Yes,33,Yes,No,DSL,No,No,Yes,Yes,Yes,Yes,One year,Yes,Mailed check,73.90,2405.05,Yes,"I've been a customer with this company for over 2 and a half years now, and unfortunately I have decided to churn. The DSL internet service was reliable for the most part, but the monthly charges of $73.9 were a bit steep for the quality of service I was receiving. Additionally, having to mail in a check for payment was inconvenient and outdated. I ultimately decided to switch to a different provider that offered better value and more modern payment options.",7634
2,2682-KEVRP,Female,1,No,No,22,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,One year,Yes,Mailed check,20.05,417.00,No,"I have been a customer with this company for 22 months now and I have been satisfied with the service. I have not experienced any issues with billing or service interruptions. The monthly charges are affordable at $20.05 and I appreciate being able to pay by mailing a check. The lack of internet service is not a problem for me as I do not require it. Overall, I am happy with my experience as a customer and have no complaints.",3628


## 2. Problemas recebidos da Bronze

A avaliação de qualidade foi feita na camada Bronze. Esta é a lista de problemas que ela entregou e o tratamento aplicado aqui:

| Ponto encontrado | Quantidade | O que fazemos |
|---|---|---|
| `PhoneService` com "yes"/"no" minúsculos | 70 | Padronizar a escrita |
| `tenure` com "unknown" | 19 | Estimar pelo total pago ÷ mensalidade (1 deles também não tem total e é removido) |
| `TotalCharges` vazio com `tenure = 0` | 10 | Preencher com 0 (cliente novo) |
| `TotalCharges` vazio nos demais | 268 | Estimar por `tenure × MonthlyCharges` |
| Nomes e valores em inglês | todas | Traduzir para português |
| Tipos errados (texto em vez de número/booleano) | várias | Corrigir a tipagem |

Antes de começar, conferimos que a tabela recebida é a mesma que a Bronze registrou.

In [2]:
import json

with open('../Data/Bronze/metadados_ingestao.json', encoding='utf-8') as arquivo:
    metadados_bronze = json.load(arquivo)

print('Linhas registradas na Bronze:', metadados_bronze['linhas'], '| linhas lidas:', len(df))
print('Ausentes em tenure (texto "unknown"):', (df['tenure'] == 'unknown').sum())
print('Ausentes em TotalCharges:', df['TotalCharges'].isna().sum())

Linhas registradas na Bronze: 5634 | linhas lidas: 5634
Ausentes em tenure (texto "unknown"): 19
Ausentes em TotalCharges: 278


## 3. Limpeza

In [3]:
# Remover espaços extras no começo e no fim dos textos
for coluna in df.select_dtypes(include=['object', 'string']).columns:
    df[coluna] = df[coluna].str.strip()

# Valores que representam "não informado" viram nulo de verdade
df = df.replace({'unknown': pd.NA, 'Unknown': pd.NA, 'null': pd.NA, 'NULL': pd.NA, '': pd.NA})

print('Valores ausentes por coluna após a limpeza:')
df.isna().sum()[lambda s: s > 0]

Valores ausentes por coluna após a limpeza:


,0
tenure,19
TotalCharges,278


### Tradução dos valores

Colocamos tudo em minúsculas antes de traduzir. Assim "Yes" e "yes" caem no mesmo valor e o problema de `PhoneService` é resolvido junto com a tradução.

In [4]:
sim_nao = {'yes': 'Sim', 'no': 'Não'}
servico_internet = {'yes': 'Sim', 'no': 'Não', 'no internet service': 'Sem serviço de internet'}

traducao_valores = {
    'gender': {'male': 'Masculino', 'female': 'Feminino'},
    'Partner': sim_nao,
    'Dependents': sim_nao,
    'PhoneService': sim_nao,
    'PaperlessBilling': sim_nao,
    'Churn': sim_nao,
    'MultipleLines': {'yes': 'Sim', 'no': 'Não', 'no phone service': 'Sem serviço de telefonia'},
    'InternetService': {'dsl': 'DSL', 'fiber optic': 'Fibra óptica', 'no': 'Sem serviço de internet'},
    'OnlineSecurity': servico_internet,
    'OnlineBackup': servico_internet,
    'DeviceProtection': servico_internet,
    'TechSupport': servico_internet,
    'StreamingTV': servico_internet,
    'StreamingMovies': servico_internet,
    'Contract': {'month-to-month': 'Mensal', 'one year': 'Um ano', 'two year': 'Dois anos'},
    'PaymentMethod': {
        'electronic check': 'Cheque eletrônico',
        'mailed check': 'Cheque pelo correio',
        'bank transfer (automatic)': 'Transferência bancária automática',
        'credit card (automatic)': 'Cartão de crédito automático'
    }
}

for coluna, mapa in traducao_valores.items():
    df[coluna] = df[coluna].str.lower().replace(mapa)

df['PhoneService'].value_counts()

,count
PhoneService,
Sim,5071
Não,563


### Tradução das colunas

A coluna de feedback continua com o texto em inglês. Traduzir textos livres pode mudar o sentido, e se o grupo fizer análise de sentimento mais pra frente, os modelos funcionam melhor no idioma original.

In [5]:
traducao_colunas = {
    'customerID': 'ID_Cliente',
    'gender': 'Genero',
    'SeniorCitizen': 'Idoso',
    'Partner': 'Conjuge',
    'Dependents': 'Dependentes',
    'tenure': 'Tempo_Assinatura',
    'PhoneService': 'Servico_Telefonia',
    'MultipleLines': 'Varias_Linhas',
    'InternetService': 'Servico_Internet',
    'OnlineSecurity': 'Seguranca_Online',
    'OnlineBackup': 'Backup_Online',
    'DeviceProtection': 'Protecao_Dispositivos',
    'TechSupport': 'Suporte_Tecnico',
    'StreamingTV': 'Streaming_TV',
    'StreamingMovies': 'Streaming_Filmes',
    'Contract': 'Contrato',
    'PaperlessBilling': 'Faturamento_Eletronico',
    'PaymentMethod': 'Forma_Pagamento',
    'MonthlyCharges': 'Valores_Mensais',
    'TotalCharges': 'Valores_Totais',
    'Churn': 'Status_Cancelamento',
    'CustomerFeedback': 'Feedback_Cliente',
    'MonthlyIncome': 'Renda_Mensal'
}
df = df.rename(columns=traducao_colunas)
df.columns.tolist()

['ID_Cliente', 'Genero', 'Idoso', 'Conjuge', 'Dependentes', 'Tempo_Assinatura', 'Servico_Telefonia', 'Varias_Linhas', 'Servico_Internet', 'Seguranca_Online', 'Backup_Online', 'Protecao_Dispositivos', 'Suporte_Tecnico', 'Streaming_TV', 'Streaming_Filmes', 'Contrato', 'Faturamento_Eletronico', 'Forma_Pagamento', 'Valores_Mensais', 'Valores_Totais', 'Status_Cancelamento', 'Feedback_Cliente', 'Renda_Mensal']

### Tipagem

- Números: tempo de assinatura (inteiro, em meses), valores e renda.
- Booleanos: colunas que só têm Sim/Não. `Idoso` já vinha como 0/1.
- Categorias: colunas com poucas opções fixas, como tipo de contrato e de internet.

In [6]:
df['Tempo_Assinatura'] = pd.to_numeric(df['Tempo_Assinatura']).astype('Int64')
df['Valores_Mensais'] = df['Valores_Mensais'].astype('Float64')
df['Valores_Totais'] = df['Valores_Totais'].astype('Float64')
df['Renda_Mensal'] = df['Renda_Mensal'].astype('Float64')

df['Idoso'] = df['Idoso'].astype('boolean')

colunas_booleanas = ['Conjuge', 'Dependentes', 'Servico_Telefonia',
                     'Faturamento_Eletronico', 'Status_Cancelamento']
for coluna in colunas_booleanas:
    df[coluna] = df[coluna].map({'Sim': True, 'Não': False}).astype('boolean')

colunas_categoricas = ['Genero', 'Servico_Internet', 'Varias_Linhas', 'Seguranca_Online',
                       'Backup_Online', 'Protecao_Dispositivos', 'Suporte_Tecnico',
                       'Streaming_TV', 'Streaming_Filmes', 'Contrato', 'Forma_Pagamento']
for coluna in colunas_categoricas:
    df[coluna] = df[coluna].astype('category')

df.dtypes

,0
ID_Cliente,str
Genero,category
Idoso,boolean
Conjuge,boolean
Dependentes,boolean
Tempo_Assinatura,Int64
Servico_Telefonia,boolean
Varias_Linhas,category
Servico_Internet,category
Seguranca_Online,category


## 4. Tratamento dos valores ausentes

A Bronze mostrou que `Valores_Totais ≈ Tempo_Assinatura × Valores_Mensais`. Usamos essa relação para preencher os vazios, em vez de usar média ou mediana da coluna, que colocariam o mesmo valor para um cliente de 2 meses e outro de 70.

Para não perder a informação de que o valor foi estimado, criamos duas colunas de controle (`Tempo_Estimado` e `Total_Estimado`). Quem quiser fazer uma análise só com os valores originais consegue filtrar por elas.

In [7]:
# Um cliente não tem nem o tempo nem o total. Sem um dos dois não há como estimar o outro,
# então ele sai da base (é 1 cliente em 5.634, não muda nenhum resultado)
sem_base = df['Tempo_Assinatura'].isna() & df['Valores_Totais'].isna()
print('Clientes removidos por não ter tempo nem total:', sem_base.sum())
df = df[~sem_base].reset_index(drop=True)

# Tempo de assinatura desconhecido: estimado pelo total pago dividido pela mensalidade.
# O resultado é limitado a 72 meses, que é o maior tempo registrado na base.
df['Tempo_Estimado'] = df['Tempo_Assinatura'].isna()
estimativa_tempo = (df['Valores_Totais'] / df['Valores_Mensais']).round().clip(upper=72)
df.loc[df['Tempo_Estimado'], 'Tempo_Assinatura'] = estimativa_tempo[df['Tempo_Estimado']].astype('Int64')

# Total vazio: cliente novo (tempo 0) fica com 0; os demais recebem tempo x mensalidade
df['Total_Estimado'] = df['Valores_Totais'].isna()
estimativa_total = (df['Tempo_Assinatura'] * df['Valores_Mensais']).round(2)
df.loc[df['Total_Estimado'], 'Valores_Totais'] = estimativa_total[df['Total_Estimado']]

print('Tempos estimados:', df['Tempo_Estimado'].sum())
print('Totais estimados:', df['Total_Estimado'].sum())
print('Ausentes restantes (sem contar o feedback):', df.drop(columns='Feedback_Cliente').isna().sum().sum())

Clientes removidos por não ter tempo nem total: 1
Tempos estimados: 18
Totais estimados: 277
Ausentes restantes (sem contar o feedback): 0


## 5. Validação final

Conferimos o número de clientes, se ainda sobrou algum ausente e se os valores estão dentro de faixas possíveis.

In [8]:
print('Linhas e colunas:', df.shape)
print('Tempo de assinatura entre', df['Tempo_Assinatura'].min(), 'e', df['Tempo_Assinatura'].max(), 'meses')
print('Mensalidade entre', df['Valores_Mensais'].min(), 'e', df['Valores_Mensais'].max())
print('Taxa geral de cancelamento:', f"{df['Status_Cancelamento'].mean():.1%}")

df.describe().round(2)

Linhas e colunas: (5633, 25)
Tempo de assinatura entre 0 e 72 meses
Mensalidade entre 18.4 e 118.65
Taxa geral de cancelamento: 26.7%


,Tempo_Assinatura,Valores_Mensais,Valores_Totais,Renda_Mensal
count,5633.0,5633.0,5633.0,5633.0
mean,32.33,64.6,2272.09,5104.95
std,24.5,30.15,2258.36,2839.48
min,0.0,18.4,0.0,100.0
25%,9.0,35.15,402.6,2693.0
50%,29.0,70.3,1390.6,5084.0
75%,55.0,89.8,3756.4,7575.0
max,72.0,118.65,8684.8,9993.0


A base terminou com 5.633 clientes (só 1 removido), sem valores ausentes, com nomes em português e tipos corretos. A taxa geral de cancelamento é de 26,7%, ou seja, pouco mais de 1 em cada 4 clientes saiu. Esse número é a referência usada nas camadas Gold para dizer se um grupo cancela mais ou menos que a média.

In [9]:
df.to_csv('../Data/Silver/tabela_silver.csv', index=False, encoding='utf-8-sig')
print('Camada Silver salva em ../Data/Silver/tabela_silver.csv')

Camada Silver salva em ../Data/Silver/tabela_silver.csv
